In [2]:
import json
import nltk
import numpy as np
import random
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder

# Download NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load intents
with open('intents.json', 'r') as f:
    intents = json.load(f)

print("✅ Libraries imported successfully!")
print("✅ Intents loaded successfully!")
print("Total intents:", len(intents['intents']))

[nltk_data] Downloading package punkt to C:\Users\MANJUNATH S
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to C:\Users\MANJUNATH S
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to C:\Users\MANJUNATH S
[nltk_data]     V\AppData\Roaming\nltk_data...


✅ Libraries imported successfully!
✅ Intents loaded successfully!
Total intents: 18


In [4]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to C:\Users\MANJUNATH S
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [5]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Prepare data
patterns = []
tags = []

for intent in intents['intents']:
    for pattern in intent['patterns']:
        # Tokenize and lemmatize each pattern
        words = nltk.word_tokenize(pattern.lower())
        words = [lemmatizer.lemmatize(w) for w in words]
        cleaned = ' '.join(words)
        patterns.append(cleaned)
        tags.append(intent['tag'])

print("✅ Data prepared successfully!")
print("Total training samples:", len(patterns))
print("Total unique intents:", len(set(tags)))
print("\nSample patterns:")
for i in range(5):
    print(f"  Pattern: '{patterns[i]}' → Tag: '{tags[i]}'")

✅ Data prepared successfully!
Total training samples: 100
Total unique intents: 18

Sample patterns:
  Pattern: 'hello' → Tag: 'greeting'
  Pattern: 'hi' → Tag: 'greeting'
  Pattern: 'hey' → Tag: 'greeting'
  Pattern: 'good morning' → Tag: 'greeting'
  Pattern: 'good evening' → Tag: 'greeting'


In [6]:
# Step 3 - Vectorize and train
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(patterns)

# Encode labels
encoder = LabelEncoder()
y = encoder.fit_transform(tags)

# Train Neural Network
classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=500,
    random_state=42
)
classifier.fit(X, y)

print("✅ Model trained successfully!")
print("Total classes:", len(encoder.classes_))
print("Classes:", list(encoder.classes_))

✅ Model trained successfully!
Total classes: 18
Classes: [np.str_('app_issue'), np.str_('bank_balance'), np.str_('block_card'), np.str_('book_appointment'), np.str_('cancel_order'), np.str_('contact_agent'), np.str_('delivery_delay'), np.str_('doctor_availability'), np.str_('goodbye'), np.str_('greeting'), np.str_('offers'), np.str_('order_status'), np.str_('payment_issue'), np.str_('refund'), np.str_('reset_password'), np.str_('return_product'), np.str_('thanks'), np.str_('working_hours')]


In [7]:
# Step 4 - Prediction function
def predict_intent(user_input):
    # Clean and tokenize input
    words = nltk.word_tokenize(user_input.lower())
    words = [lemmatizer.lemmatize(w) for w in words]
    cleaned = ' '.join(words)
    
    # Vectorize
    X_input = vectorizer.transform([cleaned])
    
    # Predict
    prediction = classifier.predict(X_input)[0]
    probability = max(classifier.predict_proba(X_input)[0])
    
    # Decode intent
    intent_tag = encoder.inverse_transform([prediction])[0]
    
    return intent_tag, probability

# Step 5 - Get response function
def get_response(intent_tag):
    for intent in intents['intents']:
        if intent['tag'] == intent_tag:
            return random.choice(intent['responses'])
    return "I'm sorry, I didn't understand that. Please try again!"

# Test it
test_input = "Where is my order?"
intent, confidence = predict_intent(test_input)
response = get_response(intent)

print(f"User Input  : {test_input}")
print(f"Intent      : {intent}")
print(f"Confidence  : {confidence*100:.2f}%")
print(f"Response    : {response}")


User Input  : Where is my order?
Intent      : order_status
Confidence  : 98.53%
Response    : You can track your order using the tracking link sent to your email.


In [8]:
# Test multiple inputs
test_cases = [
    "Hello there",
    "My payment failed",
    "I want to return my product",
    "Block my debit card",
    "Book a doctor appointment",
    "App is not working",
    "Any discount available?",
    "I need to talk to a human",
    "Thank you so much",
    "Bye"
]

print("=" * 65)
print(f"{'User Input':<35} {'Intent':<25} {'Confidence'}")
print("=" * 65)

for test in test_cases:
    intent, confidence = predict_intent(test)
    print(f"{test:<35} {intent:<25} {confidence*100:.2f}%")

User Input                          Intent                    Confidence
Hello there                         greeting                  99.52%
My payment failed                   payment_issue             99.09%
I want to return my product         return_product            99.37%
Block my debit card                 block_card                99.87%
Book a doctor appointment           book_appointment          99.88%
App is not working                  app_issue                 98.33%
Any discount available?             offers                    99.72%
I need to talk to a human           contact_agent             99.74%
Thank you so much                   thanks                    90.92%
Bye                                 goodbye                   99.18%
